# Task 2.1 — Classificatore Naive Bayes costruito a mano

**Obiettivo del Task 2** (consegna): *definire manualmente uno o due classificatori sul file
`manuale.csv`, adottando uno o due modelli illustrati a lezione, e valutarne le prestazioni sullo
stesso file. Dopo aver illustrato i passi per adattare i modelli ai dati, implementare i
classificatori in Python, utilizzando eventualmente delle API.*

Questo notebook copre il **primo** dei due classificatori richiesti (siamo un gruppo di due
componenti); il secondo, l'**albero di decisione**, e' in `02.2_albero_decisione.ipynb`.

I due modelli sono stati scelti perche' sono complementari, e quindi il confronto del Task 4 e'
informativo invece che ridondante:

| | albero di decisione | Naive Bayes |
|---|---|---|
| tipo | discriminativo | generativo |
| uso delle feature | una alla volta, in sequenza | tutte insieme, con pari peso |
| output | regole esplicite | probabilita' a posteriori |
| ipotesi forte | la classe si separa a soglie | indipendenza condizionale delle feature |

Le motivazioni discorsive sono in `documentation/02.1_naive_bayes.md`.

In [1]:
import numpy as np
import pandas as pd

SEME = 42          # coerente con il Task 1

manuale = pd.read_csv("../data/manuale.csv").set_index("USERID")
y = manuale["ABBANDONO"]
X = manuale.drop(columns="ABBANDONO")

print("campioni:", len(manuale), " feature:", X.shape[1])
print("distribuzione delle classi:", y.value_counts().to_dict())
manuale

campioni: 12  feature: 8
distribuzione delle classi: {0: 6, 1: 6}


,n_azioni,n_attivita_distinte,n_giorni_attivi,durata_giorni,feature0_media,feature1_media,feature2_media,feature3_media,ABBANDONO
USERID,,,,,,,,,
3969,44,26,5,4.941,-0.167,0.258,-0.053,-0.013,0
635,138,73,7,26.158,0.081,0.228,-0.093,-0.044,0
4344,5,3,1,0.000,-0.320,-0.436,0.107,-0.067,1
1714,97,40,11,27.117,0.014,-0.200,-0.012,-0.057,0
1123,8,5,1,0.013,-0.320,-0.118,0.044,-0.067,1
3637,6,5,1,0.082,-0.320,-0.012,0.023,-0.067,1
4413,94,19,5,5.156,-0.191,-0.409,0.096,-0.031,1
2535,65,42,6,25.987,1.215,-0.436,0.045,0.831,0
4200,7,5,2,3.618,-0.320,0.291,0.035,-0.067,1


## 1. Il modello

Il Naive Bayes applica il teorema di Bayes per stimare la probabilita' di ogni classe dato il
vettore di feature:

$$P(y \mid x_1, \dots, x_n) = \frac{P(y) \cdot P(x_1, \dots, x_n \mid y)}{P(x_1, \dots, x_n)}$$

Il denominatore e' lo stesso per entrambe le classi, quindi per **scegliere** la classe si puo'
ignorare. L'ipotesi *naive* e' che le feature siano **condizionatamente indipendenti** data la
classe, il che spezza la verosimiglianza congiunta in un prodotto di termini semplici:

$$\hat{y} = \arg\max_y \; P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

Servono quindi due sole quantita': le **probabilita' a priori** `P(y)` e le **probabilita'
condizionate** `P(x_i | y)`. Entrambe si stimano contando.

In [2]:
priori = {k: float((y == k).mean()) for k in (0, 1)}
print("probabilita' a priori:")
for k, p in priori.items():
    print(f"  P(ABBANDONO = {k}) = {(y == k).sum()}/{len(y)} = {p:.4f}")

probabilita' a priori:
  P(ABBANDONO = 0) = 6/12 = 0.5000
  P(ABBANDONO = 1) = 6/12 = 0.5000


**Osservazione.** I priori valgono 0,5 per costruzione: il campionamento del Task 1 era
stratificato, quindi `manuale.csv` contiene 6 abbandoni e 6 non-abbandoni. Il modello non ha alcuna
preferenza a priori per una classe, e la decisione dipendera' **solo** dalle verosimiglianze.

## 2. Adattare il modello a feature continue: discretizzazione alla mediana

Il Naive Bayes visto a lezione stima `P(x_i | y)` **contando** quante volte ogni valore compare in
ogni classe. Le nostre otto feature sono pero' continue: ogni valore compare una volta sola, quindi
i conteggi grezzi sarebbero tutti 1 o 0 e non stimerebbero niente.

Le trasformiamo allora in variabili binarie con **una sola regola, uguale per tutte**: ogni feature
viene tagliata alla **propria mediana**, calcolata sui 12 campioni.

- `1` = valore **sopra** la mediana della feature
- `0` = valore **sotto o pari** alla mediana

E' una regola sola da giustificare, non otto soglie scelte a occhio; e la mediana e' il taglio che
per definizione bilancia il numero di campioni ai due lati, quindi non privilegia nessuna classe.
Nell'appendice al punto 8 mostriamo l'alternativa considerata — il Naive Bayes **gaussiano** — e
perche' e' stata scartata **su questi dati**.

In [3]:
SOGLIE = X.median()                  # una soglia per feature, stimata sui 12 campioni
Xb = (X > SOGLIE).astype(int)        # 1 = sopra la mediana

print("soglie usate (mediane su manuale.csv):")
for f, s in SOGLIE.items():
    alti = int(Xb[f].sum())
    print(f"  {f:22s} mediana = {s:8.3f}   -> {alti} campioni sopra, {len(Xb) - alti} sotto o pari")

Xb.assign(ABBANDONO=y)

soglie usate (mediane su manuale.csv):
  n_azioni               mediana =   54.500   -> 6 campioni sopra, 6 sotto o pari
  n_attivita_distinte    mediana =   22.500   -> 6 campioni sopra, 6 sotto o pari
  n_giorni_attivi        mediana =    5.000   -> 5 campioni sopra, 7 sotto o pari
  durata_giorni          mediana =    5.048   -> 6 campioni sopra, 6 sotto o pari
  feature0_media         mediana =   -0.124   -> 6 campioni sopra, 6 sotto o pari
  feature1_media         mediana =   -0.065   -> 6 campioni sopra, 6 sotto o pari
  feature2_media         mediana =    0.005   -> 6 campioni sopra, 6 sotto o pari
  feature3_media         mediana =   -0.051   -> 6 campioni sopra, 6 sotto o pari


,n_azioni,n_attivita_distinte,n_giorni_attivi,durata_giorni,feature0_media,feature1_media,feature2_media,feature3_media,ABBANDONO
USERID,,,,,,,,,
3969,0,1,0,0,0,1,0,1,0
635,1,1,1,1,1,1,0,1,0
4344,0,0,0,0,0,0,1,0,1
1714,1,1,1,1,1,0,0,0,0
1123,0,0,0,0,0,0,1,0,1
3637,0,0,0,0,0,1,1,0,1
4413,1,0,0,1,0,0,1,1,1
2535,1,1,1,1,1,0,1,1,0
4200,0,0,0,0,0,1,1,0,1


**Nota sui pareggi.** Con 12 campioni la mediana e' la media fra il 6° e il 7° valore ordinato. Se i
due coincidono la mediana e' un valore **osservato**, e i campioni pari a quel valore finiscono nel
gruppo «sotto o pari»: e' il caso di `n_giorni_attivi`, dove la divisione risulta 7/5 invece che
6/6. Non e' un problema — la regola resta la stessa per tutti — ma va detto, perche' spiega perche'
non tutte le feature si dividono esattamente a meta'.

## 3. Probabilita' condizionate e correzione di Laplace

Per ogni feature e ogni classe contiamo quanti campioni stanno sopra la mediana. La stima diretta
sarebbe `P(x=1 | y) = conteggio / totale della classe`, ma con **6 campioni per classe** e' facile
ottenere uno 0: basta che nessun abbandono stia sopra la mediana di una feature.

Uno zero nel prodotto azzera l'intera verosimiglianza, e quella classe diventa impossibile
qualunque cosa dicano le altre sette feature. Per evitarlo usiamo la **correzione di Laplace**, che
aggiunge un'osservazione fittizia per ciascuno dei due valori possibili:

$$P(x_i = 1 \mid y) = \frac{\text{conteggio} + 1}{n_y + 2}$$

In [4]:
def tabella_contingenza(Xb, y):
    """Per ogni feature e ogni classe: conteggi grezzi e probabilita' con correzione di Laplace."""
    righe = []
    for c in Xb.columns:
        for k in (0, 1):
            sub = Xb.loc[y == k, c]
            alti, n = int(sub.sum()), len(sub)
            righe.append({"feature": c, "classe": k,
                          "sopra": alti, "sotto": n - alti,
                          "P(sopra|classe) grezza": alti / n,
                          "P(sopra|classe) Laplace": (alti + 1) / (n + 2)})
    return pd.DataFrame(righe)

contingenza = tabella_contingenza(Xb, y)

estreme = contingenza[contingenza["P(sopra|classe) grezza"].isin([0.0, 1.0])]
print(f"celle con stima grezza 0 o 1: {len(estreme)} su {len(contingenza)}")
print(estreme[["feature", "classe", "sopra", "sotto",
               "P(sopra|classe) grezza", "P(sopra|classe) Laplace"]].to_string(index=False))
print(f"\nvalori possibili dopo Laplace su 6 campioni: {1/8:.3f} e {7/8:.3f}")

contingenza.round(4)

celle con stima grezza 0 o 1: 3 su 16
            feature  classe  sopra  sotto  P(sopra|classe) grezza  P(sopra|classe) Laplace
n_attivita_distinte       0      6      0                     1.0                    0.875
n_attivita_distinte       1      0      6                     0.0                    0.125
    n_giorni_attivi       1      0      6                     0.0                    0.125

valori possibili dopo Laplace su 6 campioni: 0.125 e 0.875


,feature,classe,sopra,sotto,P(sopra|classe) grezza,P(sopra|classe) Laplace
0,n_azioni,0,5,1,0.8333,0.750
1,n_azioni,1,1,5,0.1667,0.250
2,n_attivita_distinte,0,6,0,1.0000,0.875
3,n_attivita_distinte,1,0,6,0.0000,0.125
4,n_giorni_attivi,0,5,1,0.8333,0.750
5,n_giorni_attivi,1,0,6,0.0000,0.125
6,durata_giorni,0,5,1,0.8333,0.750
7,durata_giorni,1,1,5,0.1667,0.250
8,feature0_media,0,5,1,0.8333,0.750
9,feature0_media,1,1,5,0.1667,0.250


**Perche' la correzione serve davvero qui.** Tre celle su sedici hanno conteggio 0 oppure 6, cioe'
stima grezza esattamente 0,0000 o 1,0000: `n_attivita_distinte` e `n_giorni_attivi` non hanno
**nessun** abbandono sopra la mediana, e `n_attivita_distinte` ha **tutti** i non-abbandoni sopra.

Sono stime impossibili da credere con sei osservazioni, e soprattutto uno 0 azzererebbe l'intero
prodotto: quella classe diventerebbe impossibile qualunque cosa dicano le altre sette feature.
Laplace porta quei valori a 1/8 = 0,125 e 7/8 = 0,875, lasciando al modello la possibilita' di
essere smentito dal resto dell'evidenza.

## 4. Il classificatore

Mettiamo insieme priori e condizionate. Il prodotto di otto probabilita' e' un numero molto
piccolo, quindi lavoriamo con i **logaritmi**: il logaritmo e' monotono, percio' la classe che
massimizza la somma dei log e' la stessa che massimizza il prodotto, ma senza rischio di
*underflow* numerico.

$$\log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)$$

In [5]:
P_sopra = {(r["feature"], r["classe"]): r["P(sopra|classe) Laplace"] for _, r in contingenza.iterrows()}

def punteggi(Xb):
    """Log-verosimiglianza di ciascuna classe, riga per riga."""
    out = {}
    for k in (0, 1):
        lp = np.full(len(Xb), np.log(priori[k]))
        for c in Xb.columns:
            p = P_sopra[(c, k)]
            lp = lp + np.log(np.where(Xb[c] == 1, p, 1 - p))
        out[f"log P(classe {k})"] = lp
    return pd.DataFrame(out, index=Xb.index)

def predici_nb(Xb):
    s = punteggi(Xb)
    return (s["log P(classe 1)"] > s["log P(classe 0)"]).astype(int)

pred = predici_nb(Xb)

(punteggi(Xb)
 .assign(margine=lambda d: (d["log P(classe 1)"] - d["log P(classe 0)"]).abs(),
         previsto=pred, vero=y, corretto=lambda d: d["vero"] == d["previsto"])
 .sort_values("margine")
 .round(3))

,log P(classe 0),log P(classe 1),margine,previsto,vero,corretto
USERID,,,,,,
3969,-7.417,-7.523,0.105,0,0,True
4413,-8.775,-6.164,2.611,1,1,True
3071,-9.874,-5.066,4.808,1,1,True
2535,-4.632,-11.155,6.523,0,0,True
1714,-4.632,-11.155,6.523,0,0,True
3637,-11.560,-3.379,8.181,1,1,True
4200,-11.560,-3.379,8.181,1,1,True
4344,-12.071,-2.869,9.203,1,1,True
1123,-12.071,-2.869,9.203,1,1,True


## 5. Prestazioni sul file `manuale.csv`

La consegna chiede di valutare il classificatore **sullo stesso file** su cui e' stato costruito.
E' una valutazione *in-sample*: dice se il modello ha imparato i dati che ha visto, non se
generalizza. Al punto 7 lo mettiamo alla prova sui 7.035 studenti di `training.csv`.

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print(f"accuratezza = {accuracy_score(y, pred):.4f}")
print(f"precisione  = {precision_score(y, pred):.4f}")
print(f"richiamo    = {recall_score(y, pred):.4f}")
print(f"F1          = {f1_score(y, pred):.4f}")
print("\nmatrice di confusione [righe = vero, colonne = previsto]:")
print(pd.DataFrame(confusion_matrix(y, pred), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))

accuratezza = 1.0000
precisione  = 1.0000
richiamo    = 1.0000
F1          = 1.0000

matrice di confusione [righe = vero, colonne = previsto]:
        prev 0  prev 1
vero 0       6       0
vero 1       0       6


## 6. Verifica con le API di scikit-learn

La consegna chiede di implementare il classificatore *"utilizzando eventualmente delle API"*.
Avendolo gia' scritto a mano, usiamo l'API come **controprova**.

Il modello corrispondente e' `BernoulliNB`, che e' esattamente il Naive Bayes su feature binarie:
con `alpha=1.0` applica la stessa correzione di Laplace che abbiamo scritto noi, cioe'
`(conteggio + 1) / (n + 2)`. Se la nostra implementazione e' corretta, deve stimare le stesse
probabilita' e produrre le stesse 12 predizioni.

In [7]:
from sklearn.naive_bayes import BernoulliNB

bnb = BernoulliNB(alpha=1.0).fit(Xb, y)

nostre = np.array([[P_sopra[(c, k)] for c in Xb.columns] for k in (0, 1)])
print("stesse P(sopra|classe):", bool(np.allclose(np.exp(bnb.feature_log_prob_), nostre)))
print("stessi priori         :", bool(np.allclose(np.exp(bnb.class_log_prior_), [priori[0], priori[1]])))
print("stesse 12 predizioni  :", bool((bnb.predict(Xb) == pred.to_numpy()).all()))
print("stesse log-probabilita':", bool(np.allclose(bnb.predict_log_proba(Xb),
      punteggi(Xb).to_numpy() - np.logaddexp.reduce(punteggi(Xb).to_numpy(), axis=1, keepdims=True))))

stesse P(sopra|classe): True
stessi priori         : True
stesse 12 predizioni  : True
stesse log-probabilita': True


## 7. Anticipazione del Task 4: quanto vale davvero questo classificatore?

Applichiamo il modello ai 7.035 studenti di `training.csv`, che per costruzione non contengono i 12
campioni di `manuale.csv`.

**Attenzione a un dettaglio che sarebbe un errore.** Le soglie di discretizzazione vanno prese
**quelle stimate su `manuale.csv`**, non ricalcolate sulle mediane di `training.csv`. Ricalcolarle
significherebbe far vedere al classificatore i dati su cui viene valutato: sarebbe un altro modello,
e il confronto perderebbe senso. Stesso principio della regola 2 del progetto sullo scaling.

In [8]:
training = pd.read_csv("../data/training.csv").set_index("USERID")
y_tr = training["ABBANDONO"]
X_tr = training.drop(columns="ABBANDONO")

Xb_tr = (X_tr > SOGLIE).astype(int)          # SOGLIE = le mediane di manuale.csv, non ricalcolate
pred_tr = predici_nb(Xb_tr)

print(f"studenti valutati: {len(y_tr)}  (positivi {y_tr.mean():.1%})")
print(f"accuratezza = {accuracy_score(y_tr, pred_tr):.4f}   F1 = {f1_score(y_tr, pred_tr):.4f}")
print(f"precisione  = {precision_score(y_tr, pred_tr):.4f}   richiamo = {recall_score(y_tr, pred_tr):.4f}")
print("\nmatrice di confusione:")
print(pd.DataFrame(confusion_matrix(y_tr, pred_tr), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))
print(f"\nbaseline 'sempre abbandono' = {accuracy_score(y_tr, np.ones(len(y_tr))):.4f}")

studenti valutati: 7035  (positivi 57.7%)
accuratezza = 0.7790   F1 = 0.8052
precisione  = 0.8193   richiamo = 0.7916

matrice di confusione:
        prev 0  prev 1
vero 0    2266     709
vero 1     846    3214

baseline 'sempre abbandono' = 0.5771


## 8. Appendice — l'alternativa scartata: Naive Bayes gaussiano

L'alternativa alla discretizzazione e' il **Naive Bayes gaussiano**, che non discretizza affatto:
assume che ogni feature, dentro ciascuna classe, segua una distribuzione normale, e stima da essa
media e varianza.

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_{i,y}^2}} \exp\left(-\frac{(x_i - \mu_{i,y})^2}{2\sigma_{i,y}^2}\right)$$

E' un modello legittimo e presente in scikit-learn (`GaussianNB`), ma **su questi dati** si comporta
male, e vale la pena mostrare perche'.

In [9]:
from sklearn.naive_bayes import GaussianNB

diagnosi = pd.DataFrame({
    "var classe 0": X[y == 0].var(ddof=0),
    "var classe 1": X[y == 1].var(ddof=0),
    "valori distinti in classe 1": X[y == 1].nunique(),
})
print("varianze stimate su 6 campioni per classe:")
print(diagnosi.round(6).to_string())

def densita(x, mu, var):
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)

mu1, var1 = X[y == 1].mean(), X[y == 1].var(ddof=0)
print("\ndensita' gaussiana della riga 4344 per la classe 1, feature per feature:")
for c in X.columns:
    print(f"  {c:22s} -> {densita(X.loc[4344, c], mu1[c], var1[c]):10.2f}")

gauss = GaussianNB().fit(X, y)
print(f"\nGaussianNB su manuale.csv : accuratezza {accuracy_score(y, gauss.predict(X)):.4f}")
print(f"GaussianNB su training.csv: accuratezza {accuracy_score(y_tr, gauss.predict(X_tr)):.4f}"
      f"   F1 {f1_score(y_tr, gauss.predict(X_tr)):.4f}")
print(f"\nil nostro NB discretizzato: accuratezza {accuracy_score(y_tr, pred_tr):.4f}"
      f"   F1 {f1_score(y_tr, pred_tr):.4f}")

varianze stimate su 6 campioni per classe:
                     var classe 0  var classe 1  valori distinti in classe 1
n_azioni              3353.583333   1025.138889                            6
n_attivita_distinte    248.888889     28.888889                            4
n_giorni_attivi          4.222222      2.000000                            3
durata_giorni           63.905648      5.168649                            6
feature0_media           0.220920      0.017271                            3
feature1_media           0.075155      0.064216                            6
feature2_media           0.002118      0.001986                            6
feature3_media           0.103837      0.000180                            2

densita' gaussiana della riga 4344 per la classe 1, feature per feature:
  n_azioni               ->       0.01
  n_attivita_distinte    ->       0.05
  n_giorni_attivi        ->       0.22
  durata_giorni          ->       0.11
  feature0_media         ->       

**Perche' lo scartiamo.** `feature3_media` ha, nella classe 1, una varianza di appena 0,00018 —
cinque abbandoni su sei hanno lo stesso identico valore, perche' e' un valore di *floor* della
feature. La gaussiana che ne risulta e' una punta strettissima, e la sua densita' vale **circa 27**
mentre le altre sette feature stanno fra 0,01 e 3,6.

Una sola feature domina cosi' l'intero prodotto, e lo fa per un motivo che non ha niente a che
vedere con l'abbandono: e' un artefatto di una varianza stimata su **sei** campioni quasi tutti
uguali. Non a caso `GaussianNB` di scikit-learn ha un parametro apposta, `var_smoothing`, che serve
proprio a impedire questo collasso.

Il conto pratico e' netto: sui 7.035 studenti di `training.csv` il gaussiano si ferma al **70,3%**,
contro il **77,9%** della versione discretizzata. La discretizzazione, che sembrava buttare via
informazione, sta in realta' buttando via **rumore**: riduce ogni feature a «sopra o sotto la
mediana», un'informazione che sei campioni per classe bastano a stimare.

## 9. Analisi critica

**Il divario in-sample / fuori campione.** Il classificatore passa dal 100% su `manuale.csv` a circa
il 78% su `training.csv`. Come per l'albero, la differenza e' il costo di aver stimato tutto —
soglie, priori e condizionate — sugli stessi dodici campioni su cui ci siamo poi misurati.

**L'ipotesi di indipendenza e' violata, e lo sappiamo.** Dal Task 1 sappiamo che `n_azioni`,
`n_attivita_distinte`, `n_giorni_attivi` e `durata_giorni` misurano tutte la lunghezza della storia
dello studente, e che le ultime due sono correlate a −0,554 con il target. Sono quindi tutt'altro
che indipendenti: il modello conta **quattro volte** la stessa evidenza, e le probabilita' a
posteriori che produce sono percio' troppo estreme. E' il difetto noto del Naive Bayes — le sue
probabilita' sono mal calibrate — che pero' danneggia poco la **decisione**, perche' per scegliere
la classe conta solo quale punteggio sia maggiore, non di quanto.

**Nessuna selezione di feature.** Abbiamo usato tutte e otto le feature. Sceglierne un
sottoinsieme «rilevante» guardando gli stessi 12 campioni sarebbe stata un'ulteriore decisione presa
in-sample, che avrebbe gonfiato il risultato sul file manuale senza alcuna garanzia sul resto. La
correzione di Laplace rende innocue le feature poco informative, che restano vicine a 0,5 in
entrambe le classi e quindi contribuiscono poco alla somma dei logaritmi.

**Limite dichiarato, ereditato dal Task 1.** Vale qui come per l'albero: la relazione «poca
attivita' → abbandono» e' in parte **tautologica**, perche' un abbandono e' per definizione la fine
dell'attivita'. Il modello e' utile solo se applicato a una finestra iniziale di osservazione, non
all'intera storia dello studente.